## Particle in a Box with Finite Walls

This section models a finite square well with box boundaries at x=0 and x=L.

Key differences from the infinite-wall model:
- The selected energy is now the **well depth** in Joules.
- Only **bound** energy levels are shown, so every displayed level satisfies $E < V_0$.
- If the well is too shallow to bind enough states, the HOMO-LUMO cell reports that clearly.

### Setup libraries and constants

In [1]:
# Import necessary libraries

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from scipy.optimize import brentq
from IPython.display import display, Markdown

In [2]:
# Physical constants (SI)
h = 6.62607015e-34      # Planck constant, J*s
m_e = 9.1093837015e-31  # Electron mass, kg
c = 2.99792458e8        # Speed of light, m/s

ANGSTROM_TO_M = 1.0e-10
HBAR = h / (2.0 * np.pi)

### Calculate finite well energies and wavefunctions

In [3]:
def finite_well_bound_states(length_angstrom, well_depth_j):
    """Return all bound-state energies for a symmetric finite square well."""
    if well_depth_j <= 0:
        return {
            "n": np.array([], dtype=int),
            "parity": [],
            "E_n": np.array([]),
            "xi": np.array([]),
            "half_width_m": 0.0,
        }

    half_width_m = 0.5 * length_angstrom * ANGSTROM_TO_M
    eta = half_width_m * np.sqrt(2.0 * m_e * well_depth_j) / HBAR
    epsilon = 1.0e-9
    roots = []
    parities = []

    def even_equation(xi):
        return xi * np.tan(xi) - np.sqrt(np.maximum(eta**2 - xi**2, 0.0))

    def odd_equation(xi):
        return -xi / np.tan(xi) - np.sqrt(np.maximum(eta**2 - xi**2, 0.0))

    def add_root(func, left, right, parity):
        if left >= right:
            return
        try:
            f_left = func(left)
            f_right = func(right)
        except FloatingPointError:
            return
        if not (np.isfinite(f_left) and np.isfinite(f_right)):
            return
        if f_left == 0:
            roots.append(left)
            parities.append(parity)
            return
        if f_left * f_right > 0:
            return
        roots.append(brentq(func, left, right))
        parities.append(parity)

    max_interval = int(np.ceil(eta / np.pi)) + 2
    for interval_index in range(max_interval):
        even_left = interval_index * np.pi + epsilon
        even_right = min(interval_index * np.pi + np.pi / 2.0 - epsilon, eta - epsilon)
        add_root(even_equation, even_left, even_right, "even")

        odd_left = interval_index * np.pi + np.pi / 2.0 + epsilon
        odd_right = min((interval_index + 1) * np.pi - epsilon, eta - epsilon)
        add_root(odd_equation, odd_left, odd_right, "odd")

    if not roots:
        return {
            "n": np.array([], dtype=int),
            "parity": [],
            "E_n": np.array([]),
            "xi": np.array([]),
            "half_width_m": half_width_m,
        }

    root_array = np.array(roots)
    energies = (HBAR**2 * root_array**2) / (2.0 * m_e * half_width_m**2)
    order = np.argsort(energies)
    energies = energies[order]
    xi_sorted = root_array[order]
    parity_sorted = [parities[index] for index in order]
    state_numbers = np.arange(1, len(energies) + 1)

    return {
        "n": state_numbers,
        "parity": parity_sorted,
        "E_n": energies,
        "xi": xi_sorted,
        "half_width_m": half_width_m,
    }


def finite_well_wavefunction(length_angstrom, well_depth_j, n_state, x_angstrom):
    """Return normalized finite-well bound-state wavefunction values psi(x)."""
    states = finite_well_bound_states(length_angstrom, well_depth_j)
    if len(states["E_n"]) == 0:
        raise ValueError("No bound states exist for the selected well depth and length")
    if n_state < 1 or n_state > len(states["E_n"]):
        raise ValueError(f"State n={n_state} is outside the available bound-state range")

    idx = n_state - 1
    E_n = states["E_n"][idx]
    parity = states["parity"][idx]
    half_width_m = states["half_width_m"]

    x_m = np.asarray(x_angstrom) * ANGSTROM_TO_M
    x_centered_m = x_m - half_width_m
    abs_centered_m = np.abs(x_centered_m)

    k = np.sqrt(2.0 * m_e * E_n) / HBAR
    kappa = np.sqrt(2.0 * m_e * (well_depth_j - E_n)) / HBAR

    inside = (x_m >= 0.0) & (x_m <= 2.0 * half_width_m)
    psi = np.zeros_like(x_m, dtype=float)

    if parity == "even":
        psi[inside] = np.cos(k * x_centered_m[inside])
        psi[~inside] = np.cos(k * half_width_m) * np.exp(-kappa * (abs_centered_m[~inside] - half_width_m))
    else:
        psi[inside] = np.sin(k * x_centered_m[inside])
        psi[~inside] = (
            np.sign(x_centered_m[~inside])
            * np.sin(k * half_width_m)
            * np.exp(-kappa * (abs_centered_m[~inside] - half_width_m))
        )

    norm = np.trapezoid(psi**2, x_m)
    if norm > 0:
        psi = psi / np.sqrt(norm)

    return psi

### Display finite well energies and wavefunctions

In [4]:
# User controls for the finite-wall energy-level display
finite_length_widget = widgets.FloatSlider(
    value=12.0,
    min=1.0,
    max=30.0,
    step=0.01,
    description="L (Angstrom)",
    continuous_update=False,
)

well_depth_widget = widgets.FloatLogSlider(
    value=2.0e-18,
    base=10,
    min=-20,
    max=-16,
    step=0.05,
    description="Depth (J)",
    readout_format=".2e",
    continuous_update=False,
)

finite_selected_level_widget = widgets.IntSlider(
    value=1,
    min=1,
    max=1,
    step=1,
    description="State n",
    continuous_update=False,
)

finite_show_state_widget = widgets.Checkbox(
    value=False,
    description="Show selected state",
)

finite_view_widget = widgets.ToggleButtons(
    options=[("Wavefunction psi", "psi"), ("Probability |psi|^2", "prob")],
    value="psi",
    description="View",
)


def plot_finite_well_levels(length_angstrom, well_depth_j, selected_n, show_state, view_mode):
    states = finite_well_bound_states(length_angstrom, well_depth_j)
    n_bound = len(states["E_n"])

    if n_bound < 1:
        print("Increase the well depth: no bound states exist for the current settings.")
        return

    selected_n = min(max(int(selected_n), 1), n_bound)

    fig, ax = plt.subplots(figsize=(7, 8))
    x_left, x_right = 0.0, length_angstrom
    x_center = 0.5 * length_angstrom
    x_fixed_half_range = 15.5
    x_min = x_center - x_fixed_half_range
    x_max = x_center + x_fixed_half_range
    y_top = well_depth_j * 1.05
    selected_energy = None

    ax.plot([x_min, x_left], [well_depth_j, well_depth_j], color="black", linewidth=2)
    ax.plot([x_left, x_left], [0.0, well_depth_j], color="black", linewidth=2)
    ax.plot([x_right, x_right], [0.0, well_depth_j], color="black", linewidth=2)
    ax.plot([x_right, x_max], [well_depth_j, well_depth_j], color="black", linewidth=2)

    for state_index, energy in enumerate(states["E_n"]):
        state_n = int(states["n"][state_index])
        if state_n == selected_n:
            selected_energy = energy
        color = "tab:red" if show_state and state_n == selected_n else "tab:green"
        ax.plot([x_left, x_right], [energy, energy], color=color, linewidth=2)
        ax.text(
            min(x_right + 0.35, x_max - 1.2),
            energy,
            f"n={state_n}",
            va="center",
            fontsize=9,
        )

    if show_state and selected_energy is not None:
        x_overlay = np.linspace(x_min, x_max, 1500)
        psi_overlay = finite_well_wavefunction(length_angstrom, well_depth_j, selected_n, x_overlay)

        if view_mode == "psi":
            max_abs_psi = float(np.max(np.abs(psi_overlay)))
            y_shape = psi_overlay / max_abs_psi if max_abs_psi > 0.0 else psi_overlay
        else:
            prob_overlay = psi_overlay**2
            max_prob = float(np.max(prob_overlay))
            y_shape = prob_overlay / max_prob if max_prob > 0.0 else prob_overlay

        overlay_amplitude = 0.06 * well_depth_j

        y_overlay = selected_energy + overlay_amplitude * y_shape
        y_top = max(y_top, float(np.max(y_overlay)) * 1.02)
        ax.plot(x_overlay, y_overlay, color="tab:purple", linewidth=1.8, alpha=0.95)

    ax.axhline(well_depth_j, color="gray", linestyle="--", linewidth=1)
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(0.0, y_top)
    ax.set_xlabel("Position x (Angstrom)")
    ax.set_ylabel("Energy (J)")
    ax.set_title(
        f"Finite-Wall Particle-in-a-Box Bound States (L = {length_angstrom:.2f} Angstrom, depth = {well_depth_j:.2e} J)"
    )
    plt.show()


def sync_finite_selected_level_widget(*_):
    states = finite_well_bound_states(finite_length_widget.value, well_depth_widget.value)
    n_bound = max(len(states["E_n"]), 1)
    finite_selected_level_widget.max = n_bound
    if finite_selected_level_widget.value > n_bound:
        finite_selected_level_widget.value = n_bound


finite_levels_out = widgets.Output()


def refresh_finite_levels(*_):
    sync_finite_selected_level_widget()
    with finite_levels_out:
        finite_levels_out.clear_output(wait=True)
        plot_finite_well_levels(
            finite_length_widget.value,
            well_depth_widget.value,
            finite_selected_level_widget.value,
            finite_show_state_widget.value,
            finite_view_widget.value,
        )


finite_length_widget.observe(sync_finite_selected_level_widget, names="value")
well_depth_widget.observe(sync_finite_selected_level_widget, names="value")
finite_length_widget.observe(refresh_finite_levels, names="value")
well_depth_widget.observe(refresh_finite_levels, names="value")
finite_selected_level_widget.observe(refresh_finite_levels, names="value")
finite_show_state_widget.observe(refresh_finite_levels, names="value")
finite_view_widget.observe(refresh_finite_levels, names="value")

sync_finite_selected_level_widget()
refresh_finite_levels()

finite_levels_panel = widgets.VBox(
    [
        widgets.HTML("<h3>Finite-Wall Energy Level Display</h3>"),
        widgets.HBox([finite_length_widget, well_depth_widget]),
        widgets.HTML("<p>Use State n to choose a level, then check 'Show selected state' to overlay the selected view on that energy line.</p>"),
        widgets.HBox([finite_selected_level_widget, finite_show_state_widget, finite_view_widget]),
        finite_levels_out,
    ]
)

display(finite_levels_panel)

### Create table of finite well energies

In [5]:

def show_finite_well_table(length_angstrom, well_depth_j):
    display(Markdown("### Finite Well Energy Levels"))
    states = finite_well_bound_states(length_angstrom, well_depth_j)
    if len(states["E_n"]) == 0:
        display(Markdown("No bound states fall below the selected well depth."))
        return

    lines = ["| n | Parity | Energy (J) |", "|---:|:---:|---:|"]
    for state_number, parity, energy in zip(states["n"], states["parity"], states["E_n"]):
        lines.append(f"| {state_number} | {parity} | {energy:.3e} |")
    display(Markdown("\n".join(lines)))


finite_table_out = widgets.Output()


def refresh_finite_well_table(*_):
    with finite_table_out:
        finite_table_out.clear_output(wait=True)
        show_finite_well_table(finite_length_widget.value, well_depth_widget.value)


finite_length_widget.observe(refresh_finite_well_table, names="value")
well_depth_widget.observe(refresh_finite_well_table, names="value")

refresh_finite_well_table()

finite_table_panel = widgets.VBox(
    [
        finite_table_out,
    ]
)

display(finite_table_panel)